# Notebook 1 — Exploratory Data Analysis & Preprocessing

## Project Overview
This notebook is the first step in building an end-to-end support ticket classification and prioritization system. The goal is to take raw customer complaint data, understand its structure, clean the text, and prepare it for machine learning.

## Dataset
**Source:** CFPB Consumer Complaint Database (via Kaggle)  
**Original size:** 2,023,066 real consumer complaints  
**Used:** 40,000 (balanced sample of 8,000 per category)

### Why this dataset?
The original plan was to use the Kaggle Customer Support Ticket dataset (suraj520). After loading and training initial models, the Logistic Regression accuracy was 20.84% — equivalent to random guessing across 5 classes. Investigation revealed that the dataset was synthetically generated with **random label assignments** — the ticket descriptions had no relationship to their category labels. No model can learn from data where labels are random.

The CFPB Consumer Complaint dataset was selected as a replacement because:
- It contains **real complaints** submitted to the US government
- Labels were assigned by **actual human reviewers**
- Text and labels are properly aligned — verified by manual inspection
- 2 million rows provides more than enough data

This decision is documented here because **data quality is more important than data quantity**.

---

In [ ]:
# Cell 1 — Imports & Setup
import os
os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
import nltk

warnings.filterwarnings('ignore')
nltk.data.path.append(r'C:\Users\Hridhayansh Reddi B\AppData\Roaming\nltk_data')

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

print('All imports successful')
print(f'Pandas version: {pd.__version__}')
print(f'Working directory: {os.getcwd()}')

---
## Step 1 — Load & Balance the Dataset

### Class balancing rationale
The raw CFPB dataset has severe class imbalance:
- Credit Reporting: 1,205,275 complaints (60% of data)
- Bank Accounts and Services: 158,640 complaints (8% of data)

Training on imbalanced data causes the model to become biased toward the majority class.

**Solution:** Sample exactly 8,000 complaints per category for 40,000 total rows with equal class representation.

---

In [ ]:
# Cell 2 — Load, Sample & Balance the Data
df_full = pd.read_csv('data/complaints.csv')

print('Full dataset size:', len(df_full))
print('\nFull category distribution:')
print(df_full['product_5'].value_counts())

samples_per_class = 8000
balanced_dfs = []

for category in df_full['product_5'].unique():
    category_df = df_full[df_full['product_5'] == category]
    n = min(len(category_df), samples_per_class)
    sampled = category_df.sample(n=n, random_state=42)
    balanced_dfs.append(sampled)

df = pd.concat(balanced_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nBalanced dataset shape: {df.shape}')
print(f'\nBalanced category distribution:')
print(df['product_5'].value_counts())
print(f'\nMissing values:')
print(df[['narrative', 'product_5']].isnull().sum())

---
## Step 2 — Exploratory Data Analysis

Before building any model we need to understand the data. EDA answers three key questions:
1. Are there data quality issues? (missing values, duplicates)
2. Are the classes balanced?
3. Does the text actually reflect the label?

---

In [ ]:
# Cell 3 — Data Quality Check
print('=== DATASET SHAPE ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing if len(missing) > 0 else 'No missing values in key columns')

print('\n=== DUPLICATE ROWS ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

print('\n=== SAMPLE NARRATIVES (verifying text-label alignment) ===')
for i in range(5):
    print(f'\n[{i}] Category: {df["product_5"][i]}')
    print(f'    Text: {df["narrative"][i][:150]}...')

In [ ]:
# Cell 4 — Visualize Category Distribution
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Consumer Complaint Distribution by Category',
             fontsize=15, fontweight='bold')

category_counts = df['product_5'].value_counts()
colors = ['steelblue', 'seagreen', 'coral', 'mediumpurple', 'goldenrod']
bars = ax.barh(category_counts.index, category_counts.values,
               color=colors, edgecolor='white')
ax.set_xlabel('Number of Complaints')
for bar, val in zip(bars, category_counts.values):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('data/distribution_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

---
## Step 3 — Text Cleaning Pipeline

Raw text cannot be fed directly into a machine learning model. We convert it into a clean, normalized form that maximizes the signal-to-noise ratio.

### Cleaning steps and rationale

| Step | What it does | Why |
|---|---|---|
| Lowercase | FRAUD → fraud | Same words treated identically regardless of capitalization |
| Remove XXXX | XXXX XXXX → empty | Anonymization placeholders carry no meaning |
| Remove emails/URLs | user@bank.com → empty | Unique strings add no classification signal |
| Remove numbers | $35.00 → empty | Specific amounts don't generalize |
| Remove punctuation | fraud! → fraud | Not meaningful for TF-IDF |
| Tokenize | Split into words | Required for stopword removal and lemmatization |
| Remove stopwords | Remove the, is, and | Common words with zero discriminative value |
| Lemmatize | charges → charge | Groups word variants, reduces vocabulary size |

### Why we did NOT over-clean
An earlier version removed domain-specific words as custom stopwords. This reduced model accuracy to near-random. TF-IDF handles common cross-category words mathematically through IDF weighting — words appearing in all documents get near-zero weight automatically.

---

In [ ]:
# Cell 5 — Prepare Text Column
df['combined_text'] = df['narrative'].astype(str)

print('Sample narratives:')
for i in range(3):
    print(f'\n[{i}] {df["product_5"][i]}')
    print(f'    {df["combined_text"][i][:200]}')

print(f'\nAverage text length: {df["combined_text"].str.split().str.len().mean():.0f} words')

In [ ]:
# Cell 6 — Text Cleaning Pipeline
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'x+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words and len(t) > 2
    ]
    return ' '.join(tokens)

df['cleaned_text'] = df['combined_text'].apply(clean_text)

print('=== BEFORE CLEANING ===')
print(df['combined_text'][0][:300])
print('\n=== AFTER CLEANING ===')
print(df['cleaned_text'][0][:300])
print(f'\nAverage cleaned text length: {df["cleaned_text"].str.split().str.len().mean():.0f} words')

In [ ]:
# Cell 7 — Word Clouds per Category
categories = df['product_5'].unique()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Most Common Words per Complaint Category',
             fontsize=16, fontweight='bold')
axes = axes.flatten()
colors = ['Blues', 'Reds', 'Greens', 'Purples', 'Oranges']

for i, category in enumerate(categories):
    text = ' '.join(df[df['product_5'] == category]['cleaned_text'].values)
    wordcloud = WordCloud(
        width=800, height=400,
        background_color='white',
        colormap=colors[i],
        max_words=50
    ).generate(text)
    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(category, fontsize=13, fontweight='bold', pad=10)
    axes[i].axis('off')

axes[5].axis('off')
plt.tight_layout()
plt.savefig('data/wordclouds_cleaned.png', dpi=150, bbox_inches='tight')
plt.show()
print('Word clouds saved.')

---
## Step 4 — Priority Label Engineering

The CFPB dataset has no priority column. Priority labels are engineered from complaint text using keyword-based rules.

| Priority | Condition | Business reasoning |
|---|---|---|
| High | 2+ high-urgency keywords | Multiple fraud/legal signals — immediate response needed |
| Medium | 1 high-urgency keyword | One urgency signal — prompt attention warranted |
| Low | Low-urgency keywords present | General inquiries — normal queue |
| Medium | No keywords match | Default fallback |

**Limitation:** This approach is extended in Notebook 3 with VADER sentiment scoring to capture emotional intensity alongside keyword presence.

---

In [ ]:
# Cell 8 — Engineer Priority Labels
def assign_priority(row):
    narrative = str(row['narrative']).lower()

    high_keywords = [
        'fraud', 'theft', 'stolen', 'unauthorized', 'identity theft',
        'scam', 'illegal', 'violation', 'discriminat', 'lawsuit',
        'attorney', 'legal action', 'incorrect information',
        'wrongful', 'harassment', 'threaten'
    ]
    low_keywords = [
        'question', 'inquiry', 'information', 'how do i', 'wondering',
        'curious', 'would like to know', 'general', 'clarification'
    ]

    high_count = sum(1 for kw in high_keywords if kw in narrative)
    low_count  = sum(1 for kw in low_keywords  if kw in narrative)

    if high_count >= 2:
        return 'High'
    elif high_count == 1:
        return 'Medium'
    elif low_count >= 1:
        return 'Low'
    else:
        return 'Medium'

df['priority'] = df.apply(assign_priority, axis=1)

print('Priority distribution:')
print(df['priority'].value_counts())

In [ ]:
# Cell 9 — Balance Priority Distribution & Save
priority_balanced = []
samples_per_priority = 7000

for priority in df['priority'].unique():
    priority_df = df[df['priority'] == priority]
    n = min(len(priority_df), samples_per_priority)
    priority_balanced.append(priority_df.sample(n=n, random_state=42))

df_final = pd.concat(priority_balanced).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print('Final priority distribution:')
print(df_final['priority'].value_counts())
print(f'\nFinal dataset shape: {df_final.shape}')

fig, ax = plt.subplots(figsize=(7, 4))
priority_counts = df_final['priority'].value_counts()
colors = {'High': '#e74c3c', 'Medium': '#f1c40f', 'Low': '#2ecc71'}
bar_colors = [colors[p] for p in priority_counts.index]
bars = ax.bar(priority_counts.index, priority_counts.values,
              color=bar_colors, edgecolor='white')
ax.set_title('Complaint Priority Distribution', fontweight='bold')
ax.set_ylabel('Number of Complaints')
for bar, val in zip(bars, priority_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 30,
            str(val), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('data/priority_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

os.makedirs('data', exist_ok=True)
df_final[['product_5', 'priority', 'combined_text', 'cleaned_text']].to_csv(
    'data/cleaned_tickets.csv', index=False
)

print(f'\nCleaned dataset saved to data/cleaned_tickets.csv')
print(f'Rows: {len(df_final)}')
print(f'\nCategory distribution:')
print(df_final['product_5'].value_counts())
print(f'\nPriority distribution:')
print(df_final['priority'].value_counts())

---
## Notebook Summary

| Step | Action | Result |
|---|---|---|
| Dataset selection | Replaced synthetic dataset with CFPB real complaints | Meaningful text-label alignment confirmed |
| Class balancing | 8,000 samples per category | 5 perfectly balanced categories |
| Text cleaning | 8-step pipeline | 205 words avg → 89 words avg after cleaning |
| Word cloud validation | Visual check of category vocabulary | Distinct domain words confirmed per category |
| Priority engineering | Keyword-based label assignment | High/Medium/Low labels created |
| Priority balancing | 7,000 samples per priority level | Balanced 3-class priority distribution |
| Save | cleaned_tickets.csv | ~21,000 rows ready for modeling |

**Next:** Notebook 2 trains category classification models on this cleaned data.

---